Set-up of the different parameters and config

In [2]:
from pathlib import Path
from datasets import (load_dataset, DatasetDict)

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

# Dataset
DATASET_NAME = "UCLNLP/adversarial_qa"
DATASET_CONFIG = "adversarialQA"

# Models (same tokenizer for both)
MODEL_BASE = "google/flan-t5-base"
MODEL_LARGE = "google/flan-t5-large"

# Prompt / length budget (from EDA)
PROMPT_TEMPLATE = "question: {question}  context: {context}"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 32
MAX_GEN_TOKENS = 48

# Reproducibility
SEED = 42
VAL_SIZE = 3000

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

PROJECT_ROOT: /Users/julianlilas/Desktop/DSTI/deep_learning/project/repo/DeepLearning
Exists: True


Split the dataset

In [ ]:
raw = load_dataset(DATASET_NAME, DATASET_CONFIG, cache_dir=DATA_RAW)

print(raw)

split = raw["train"].train_test_split(test_size=VAL_SIZE, seed=SEED)

ds = DatasetDict({
    "train": split["train"],
    "val":   split["test"],
    "test":  raw["validation"],
})

for name, dset in ds.items():
    n_empty = sum(1 for a in dset["answers"] if len(a["text"]) == 0)
    print(f"{name:6s} {len(dset):6d} rows | {n_empty:4d} without answer")

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 30000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 3000
    })
})
train   27000 rows |    0 without answer
val      3000 rows |    0 without answer
test     3000 rows |    0 without answer


We need to define the template for the input of the model when we'll fine-tune the model

In [5]:
def format_example(example):
    return {
        "input_text": PROMPT_TEMPLATE.format(
            question=example["question"],
            context=example["context"],
        ),
        "target_text": example["answers"]["text"][0],
    }

ds = ds.map(format_example)

ex = ds["train"][0]
print("INPUT:", ex["input_text"], sep="\n")
print("TARGET:", ex["target_text"], sep="\n")

INPUT:
question: What system is comparable to the xbox 360?  context: Virtually all console gaming systems of the previous generation used microprocessors developed by IBM. The Xbox 360 contains a PowerPC tri-core processor, which was designed and produced by IBM in less than 24 months. Sony's PlayStation 3 features the Cell BE microprocessor designed jointly by IBM, Toshiba, and Sony. IBM also provided the microprocessor that serves as the heart of Nintendo's new Wii U system, which debuted in 2012. The new Power Architecture-based microprocessor includes IBM's latest technology in an energy-saving silicon package. Nintendo's seventh-generation console, Wii, features an IBM chip codenamed Broadway. The older Nintendo GameCube utilizes the Gekko processor, also designed by IBM.
TARGET:
Nintendo's new Wii U system


We need now to define the tokenize function, and add all the tokenized inputs and outputs to the dataset

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)

def tokenize_example(batch):
    model_inputs = tokenizer(batch["input_text"], max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch["target_text"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

ds = ds.map(tokenize_example)

ex = ds["train"][0]
print("TARGET TEXT :", ex["target_text"])
print("LABELS DECODED:", tokenizer.decode(ex["labels"], skip_special_tokens=True))
print("INPUT LEN:", len(ex["input_ids"]), "| LABEL LEN:", len(ex["labels"]))

Map:   0%|          | 0/27000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 27000
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
})

Finally, we need to save the different batchs (train, val, test)

In [27]:
ds.save_to_disk(DATA_PROCESSED)

from datasets import load_from_disk
reloaded = load_from_disk(DATA_PROCESSED)
print(reloaded)

Saving the dataset (0/1 shards):   0%|          | 0/27000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 27000
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
})
